In [1]:

# =====================================================================
# CELL 1 - ENVIRONMENT SETUP, IMPORTS, CONFIGURATION & DEBUG UTILITIES
# =====================================================================
# Paper: Zhang et al. (2023), "Subject-independent EEG classification
# based on a hybrid neural network", Frontiers in Neuroscience.
#
# This notebook implements:
#   1) BCI Competition IV-2a data handling
#   2) 1-38 Hz preprocessing + 10-band filter bank
#   3) OVR-FBCSP + LASSO sparse spatial filters
#   4) FBGAN: generator + raw EEG discriminator + sparse-FB discriminator
#   5) CRNN-DF: spatial CNN + 2-layer LSTM + discriminative feature loss
#   6) LOSO subject-independent training and evaluation
#
# NOTE:
# The source paper reports some tensor dimensions (e.g. the GAN FC=750
# input and generator FC=256,000) without explicitly specifying every
# padding/reshape detail. The implementation below preserves all stated
# kernel/stride/channel specifications and uses non-learnable shape
# adapters where required to make the published dimensions executable.
#
# Default execution is a small SMOKE TEST so the notebook runs on a laptop.
# Set RUN_FULL_EXPERIMENT=True in CELL 1 for the full 9-subject protocol.

from __future__ import annotations

import os
import math
import json
import time
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import scipy
from scipy import signal, linalg
from scipy.io import loadmat

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.manifold import TSNE

try:
    import mne
    MNE_AVAILABLE = True
except Exception as exc:
    MNE_AVAILABLE = False
    mne = None
    warnings.warn(f"MNE not available: {exc}")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch:", torch.__version__)
print("SciPy:", scipy.__version__)
print("MNE available:", MNE_AVAILABLE)

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Device:", DEVICE)

# ---------------------------- Global config ---------------------------

SEED = 42
N_SUBJECTS = 9
N_SESSIONS = 2
N_CHANNELS = 22
FS = 250
TRIAL_SECONDS = 4
N_TIME = FS * TRIAL_SECONDS
N_CLASSES = 4
CLASS_NAMES = ["Left Hand", "Right Hand", "Both Feet", "Tongue"]

BANDS = [
    (1.0, 4.0), (4.0, 8.0), (8.0, 12.0), (12.0, 16.0), (16.0, 20.0),
    (20.0, 24.0), (24.0, 28.0), (28.0, 32.0), (32.0, 35.0), (35.0, 38.0)
]

# Paper settings
GAN_LATENT_DIM = 1600
GAN_LR = 1e-4
GAN_BATCH_SIZE = 5
CLS_LR = 1e-4
CLS_BATCH_SIZE = 32
CENTER_LAMBDA = 0.1
CENTER_SHIFT_ALPHA = 0.02
CENTER_SHIFT_EVERY = 15
CSP_M = 4
LASSO_EPS = 1e-8

# Laptop-safe execution switches
RUN_FULL_EXPERIMENT = False
RUN_SMOKE_EXPERIMENT = True
SMOKE_SUBJECTS = 2
SMOKE_SESSIONS = 1
SMOKE_TRIALS_PER_SESSION = 48
SMOKE_GAN_EPOCHS = 1
SMOKE_GAN_STEPS_PER_EPOCH = 2
SMOKE_CLS_EPOCHS = 2
FULL_GAN_EPOCHS = 50
FULL_CLS_EPOCHS = 100
DEFAULT_N_AUG = 24  # Smoke test only; full paper protocol uses 3000 = 750/class.

DATA_ROOT = Path("./BCI_Competition_IV_2a")
OUTPUT_ROOT = Path("./zhang2023_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = SEED) -> None:
    """Reproducible NumPy/PyTorch/Python seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

def print_ascii(title: str, diagram: str) -> None:
    print("\n" + "=" * 78)
    print(f"{title}")
    print("=" * 78)
    print(diagram.strip("\n"))
    print("=" * 78 + "\n")

class ShapeTracker:
    """Optional runtime shape tracing utility used by neural modules."""
    def __init__(self, debug: bool = False, name: str = "module"):
        self.debug = debug
        self.name = name

    def __call__(self, tag: str, x: torch.Tensor) -> torch.Tensor:
        if self.debug:
            print(f"[ShapeTracker:{self.name}] {tag:<22} {tuple(x.shape)}")
        return x

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def to_numpy(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

print("Configuration loaded.")
print("Input trial shape: (22, 1000)")
print("LOSO subjects:", [f"A{i}" for i in range(1, N_SUBJECTS + 1)])


PyTorch: 2.10.0
SciPy: 1.15.3
MNE available: True
Device: mps
Configuration loaded.
Input trial shape: (22, 1000)
LOSO subjects: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9']


In [2]:

# =====================================================================
# CELL 2 - ASCII ARCHITECTURE INFOGRAPHICS & MATHEMATICAL OVERVIEW
# =====================================================================

print_ascii(
    "PAPER PIPELINE - OVERVIEW",
    r"""
Subject-specific target EEG (C=22, T=1000)
                    │
                    ▼
        NaN replacement + Butterworth
            1-38 Hz, order=5
                    │
                    ▼
      Training-statistics z-score only
                    │
                    ▼
         ┌─────────────────────────┐
         │  10-band filter bank    │
         │  1-4 ... 35-38 Hz       │
         └─────────────────────────┘
                    │
                    ▼
       OVR CSP per class and band
       4 classes × 4 filters × 10
                    │
                    ▼
             160 CSP features
                    │
                    ▼
               LASSO
                    │
                    ▼
       Sparse W_csp + dynamic Var
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
      FBGAN Gθ            CRNN-DF
          │                   │
     ┌────┴─────┐        Spatial CNN
     ▼          ▼             │
    Dφ         Dψ             ▼
 raw EEG     sparse FB      2 × LSTM64
     │          │             │
     └────┬─────┘             ▼
          ▼             discriminative
      fake EEG             feature loss
          │                   │
          └────────┬──────────┘
                   ▼
         LOSO target-subject test
                   │
        ┌──────────┼───────────┐
        ▼          ▼           ▼
    Accuracy   Confusion      t-SNE
                 matrix      features
"""
)

print_ascii(
    "MATHEMATICAL FORMULATIONS",
    r"""
(1) Standardization
    X' = (X - μ_train) / (σ_train + ε)

(2) Normalized class covariance
    Rc = (1/Nc) Σ_i [ Xi Xi^T / tr(Xi Xi^T) ]

(3) OVR generalized eigenproblem
    Rc w = λ R_rest w
    Keep m=4 largest-eigenvalue columns per class/band.

(4) Candidate CSP feature matrix
    F ∈ R^(N × 160)
    (10 bands × 4 OVR classes × 4 filters)

(5) LASSO
    β* = argminβ [ (1/N)||y - β0 - Fβ||² + λ||β||1 ]
    Select non-zero coefficients and corresponding W_csp columns.

(6) Sparse spatial filtering
    Z = W_csp^T X'
    (implementation applies each retained filter to its associated
     sub-band and concatenates the resulting spatial channels)

(7) Central distance loss
    L_cen = (1/b) Σ_i ||v_i^k - cen_{y_i}^k||²

(8) Center initialization
    cen_j^0 = class-wise mean of initial feature vectors

(9) Center repulsion / shift every 15 epochs
    v_c^k = (1/C) Σ_j cen_j^k
    cen_j^(k+1) = cen_j^k +
        α * (cen_j^k - v_c^k) / ||cen_j^k - v_c^k||

(10) Joint loss
    L = CrossEntropy + λ_center * L_cen
"""
)

print(
    "Source checkpoints: the paper reports CRNN-DF mean=63.52±10.70% "
    "without augmentation and 72.82±10.44% with 3000 FBGAN samples in Table 4. "
    "The abstract reports 72.74±10.44%; the notebook preserves the paper's "
    "reported figures rather than silently reconciling them."
)



PAPER PIPELINE - OVERVIEW
Subject-specific target EEG (C=22, T=1000)
                    │
                    ▼
        NaN replacement + Butterworth
            1-38 Hz, order=5
                    │
                    ▼
      Training-statistics z-score only
                    │
                    ▼
         ┌─────────────────────────┐
         │  10-band filter bank    │
         │  1-4 ... 35-38 Hz       │
         └─────────────────────────┘
                    │
                    ▼
       OVR CSP per class and band
       4 classes × 4 filters × 10
                    │
                    ▼
             160 CSP features
                    │
                    ▼
               LASSO
                    │
                    ▼
       Sparse W_csp + dynamic Var
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
      FBGAN Gθ            CRNN-DF
          │                   │
     ┌────┴─────┐        Spatial CNN
     ▼          ▼         

In [3]:

# =====================================================================
# CELL 3 - SYNTHETIC DATA GENERATOR & REAL BCI IV-2a DATA LOADERS
# =====================================================================

print_ascii(
    "DATA MODULE",
    r"""
BCI IV-2a expected layout
  9 subjects × 2 sessions × 288 trials × 22 × 1000

                 ┌─────────────┐
                 │ subject A1  │
                 │ session 1/2 │
                 └──────┬──────┘
                        ...
                 ┌─────────────┐
                 │ subject A9  │
                 │ session 1/2 │
                 └──────┬──────┘
                        ▼
       X: (N, 22, 1000), y: (N,), subject: (N,)

Real .gdf:
  MNE Raw -> events -> cue at t=2 s -> [2,6) s -> 22×1000

Real .npz:
  expected X / y, optional subject / session keys

Real .mat:
  flexible recursive discovery of the first compatible arrays
"""
)

def generate_synthetic_trial(
    label: int,
    channels: int = N_CHANNELS,
    time_points: int = N_TIME,
    fs: int = FS,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """Generate one realistic-looking 22×1000 MI-style synthetic trial."""
    rng = rng or np.random.default_rng()
    t = np.arange(time_points) / fs
    x = 0.15 * rng.standard_normal((channels, time_points))

    # Deterministic class-dependent rhythm templates:
    # left/right ~ mu/beta modulation, feet/tongue with distinct frequencies.
    class_freqs = [10.0, 11.0, 13.0, 8.0]
    freq = class_freqs[label]

    # Approximate sensorimotor topography, not intended as clinical simulation.
    centers = {
        0: [6, 7, 8],       # left-hand-like
        1: [18, 19, 20],    # right-hand-like
        2: [10, 11, 12, 13],
        3: [14, 15, 16],
    }
    active = centers[label]
    envelope = 1.0 + 0.4 * np.sin(2 * np.pi * 0.5 * t)

    for ch in active:
        phase = rng.uniform(0, 2 * np.pi)
        amp = rng.uniform(0.35, 0.65)
        x[ch] += amp * envelope * np.sin(2 * np.pi * freq * t + phase)
        x[ch] += 0.15 * np.sin(2 * np.pi * (freq + 8) * t + phase / 2)

    # Small shared low-frequency component and channel offsets.
    x += 0.03 * np.sin(2 * np.pi * 1.5 * t)[None, :]
    x += rng.normal(0.0, 0.02, size=(channels, 1))
    return x.astype(np.float32)

def generate_synthetic_bci2a(
    subjects: int = 9,
    sessions: int = 2,
    trials_per_session: int = 288,
    seed: int = SEED,
    lazy: bool = False,
):
    """
    Synthetic BCI IV-2a-compatible generator.

    Exact full logical size:
        9 × 2 × 288 × 22 × 1000

    Full materialization is intentionally NOT done by default because it
    is about 1.1 GB as float32 before intermediate filter-bank tensors.
    Set lazy=False only when you have sufficient RAM.
    """
    total = subjects * sessions * trials_per_session
    if lazy:
        return SyntheticBCI2aGenerator(
            subjects=subjects,
            sessions=sessions,
            trials_per_session=trials_per_session,
            seed=seed,
        )

    approx_gb = total * N_CHANNELS * N_TIME * 4 / (1024**3)
    if approx_gb > 4:
        warnings.warn(
            f"Requested synthetic array is approximately {approx_gb:.2f} GiB. "
            "Consider lazy=True."
        )

    rng = np.random.default_rng(seed)
    X = np.empty((total, N_CHANNELS, N_TIME), dtype=np.float32)
    y = np.empty(total, dtype=np.int64)
    subjects_arr = np.empty(total, dtype=np.int64)
    sessions_arr = np.empty(total, dtype=np.int64)

    idx = 0
    for s in range(subjects):
        for sess in range(sessions):
            labels = np.tile(np.arange(N_CLASSES), trials_per_session // N_CLASSES)
            if len(labels) < trials_per_session:
                labels = np.resize(labels, trials_per_session)
            rng.shuffle(labels)
            for label in labels:
                X[idx] = generate_synthetic_trial(int(label), rng=rng)
                y[idx] = int(label)
                subjects_arr[idx] = s + 1
                sessions_arr[idx] = sess + 1
                idx += 1

    return X, y, subjects_arr, sessions_arr

class SyntheticBCI2aGenerator:
    """Lazy generator for the full logical 9×2×288 protocol."""
    def __init__(self, subjects=9, sessions=2, trials_per_session=288, seed=SEED):
        self.subjects = subjects
        self.sessions = sessions
        self.trials_per_session = trials_per_session
        self.seed = seed
        self.shape = (
            subjects * sessions * trials_per_session,
            N_CHANNELS,
            N_TIME,
        )

    def batches(self, batch_size: int = 32):
        rng = np.random.default_rng(self.seed)
        for subject in range(1, self.subjects + 1):
            for session in range(1, self.sessions + 1):
                labels = np.tile(
                    np.arange(N_CLASSES),
                    int(np.ceil(self.trials_per_session / N_CLASSES)),
                )[: self.trials_per_session]
                rng.shuffle(labels)
                for start in range(0, self.trials_per_session, batch_size):
                    batch_labels = labels[start:start + batch_size]
                    Xb = np.stack(
                        [generate_synthetic_trial(int(y), rng=rng) for y in batch_labels]
                    )
                    yield Xb, batch_labels.astype(np.int64), np.full(
                        len(batch_labels), subject, dtype=np.int64
                    ), np.full(len(batch_labels), session, dtype=np.int64)

def _normalize_loaded_arrays(X, y, subjects=None, sessions=None):
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    if X.ndim == 4:
        # Accept either (subjects, trials, channels, time) or
        # (trials, channels, time, 1); normalize toward (N,C,T).
        if X.shape[-1] == 1 and X.shape[2] >= 900:
            X = np.squeeze(X, -1)
        elif X.shape[1] == N_CHANNELS and X.shape[2] == N_TIME:
            X = X.reshape(-1, N_CHANNELS, N_TIME)
        else:
            X = X.reshape(-1, X.shape[-2], X.shape[-1])
    if X.ndim != 3:
        raise ValueError(f"Expected X with 3/4 dimensions, got {X.shape}.")

    if X.shape[1:] != (N_CHANNELS, N_TIME):
        raise ValueError(
            f"Expected trial shape (22,1000), got {X.shape[1:]}. "
            "Slice the input to 4 seconds at 250 Hz first."
        )

    if len(y) != len(X):
        y = np.resize(y, len(X))

    if subjects is None:
        subjects = np.ones(len(X), dtype=np.int64)
    else:
        subjects = np.asarray(subjects).reshape(-1)
        if len(subjects) != len(X):
            subjects = np.repeat(subjects, len(X) // len(subjects))

    if sessions is None:
        sessions = np.ones(len(X), dtype=np.int64)
    else:
        sessions = np.asarray(sessions).reshape(-1)
        if len(sessions) != len(X):
            sessions = np.resize(sessions, len(X))

    # Map labels to 0..3 if they are 1..4.
    unique = np.unique(y)
    if set(unique.tolist()) == {1, 2, 3, 4}:
        y = y - 1
    return X.astype(np.float32), y.astype(np.int64), subjects.astype(np.int64), sessions.astype(np.int64)

def load_npz_bci2a(path: str):
    data = np.load(path, allow_pickle=True)
    X_key = next((k for k in ["X", "eeg", "data", "signals"] if k in data), None)
    y_key = next((k for k in ["y", "labels", "target"] if k in data), None)
    if X_key is None or y_key is None:
        raise KeyError("NPZ must contain X/eeg/data/signals and y/labels/target.")
    return _normalize_loaded_arrays(
        data[X_key],
        data[y_key],
        data["subjects"] if "subjects" in data else None,
        data["sessions"] if "sessions" in data else None,
    )

def _find_mat_array(obj, predicate):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key.startswith("__"):
                continue
            result = _find_mat_array(value, predicate)
            if result is not None:
                return result
    elif isinstance(obj, np.ndarray) and predicate(obj):
        return obj
    return None

def load_mat_bci2a(path: str):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    X = _find_mat_array(
        mat,
        lambda a: a.ndim in (3, 4) and (
            N_CHANNELS in a.shape or N_TIME in a.shape
        ),
    )
    y = _find_mat_array(
        mat,
        lambda a: np.asarray(a).ndim <= 2 and np.asarray(a).size >= 4,
    )
    if X is None or y is None:
        raise ValueError(
            "Could not automatically identify EEG X/y arrays in MAT file. "
            "Use the notebook helper on your exact MAT structure."
        )
    return _normalize_loaded_arrays(X, y)

def load_gdf_bci2a(path: str):
    if not MNE_AVAILABLE:
        raise ImportError("MNE is required for .gdf loading.")

    raw = mne.io.read_raw_gdf(path, preload=True, verbose=False)
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    # BCI IV-2a labels commonly occur as 7,8,9,10 or 769,770,771,772.
    mapping_candidates = [
        {"7": 0, "8": 1, "9": 2, "10": 3},
        {"769": 0, "770": 1, "771": 2, "772": 3},
    ]
    inv_event_id = {int(v): k for k, v in event_id.items()}

    chosen = None
    for cand in mapping_candidates:
        if all(int(k) in inv_event_id for k in cand):
            chosen = cand
            break
    if chosen is None:
        # Try event-id annotation names that contain T1/T2/T3/T4.
        chosen = {}
        for code, name in inv_event_id.items():
            if str(name).endswith("T1"):
                chosen[str(code)] = 0
            elif str(name).endswith("T2"):
                chosen[str(code)] = 1
            elif str(name).endswith("T3"):
                chosen[str(code)] = 2
            elif str(name).endswith("T4"):
                chosen[str(code)] = 3
        if len(chosen) != 4:
            raise ValueError(
                f"Could not infer 4 MI event codes. event_id={event_id}"
            )

    sfreq = raw.info["sfreq"]
    start_offset = int(round(2.0 * sfreq))
    length = int(round(4.0 * sfreq))
    data = raw.get_data()

    trials = []
    labels = []
    trial_times = []
    for sample, _, code in events:
        if str(code) not in chosen:
            continue
        start = int(sample) + start_offset
        stop = start + length
        if start < 0 or stop > data.shape[1]:
            continue
        trial = data[:N_CHANNELS, start:stop]
        if trial.shape == (N_CHANNELS, N_TIME):
            trials.append(trial)
            labels.append(chosen[str(code)])
            trial_times.append(sample / sfreq)

    if not trials:
        raise RuntimeError("No valid 22×1000 MI trials found in GDF file.")

    X = np.stack(trials).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    return _normalize_loaded_arrays(X, y)

def load_bci2a_file(path: str):
    """Dispatch .gdf/.mat/.npz loader."""
    suffix = Path(path).suffix.lower()
    if suffix == ".gdf":
        return load_gdf_bci2a(path)
    if suffix == ".mat":
        return load_mat_bci2a(path)
    if suffix == ".npz":
        return load_npz_bci2a(path)
    raise ValueError(f"Unsupported file type: {suffix}")

# ------------------------------ Smoke data ----------------------------

if RUN_SMOKE_EXPERIMENT:
    X_demo, y_demo, subj_demo, sess_demo = generate_synthetic_bci2a(
        subjects=SMOKE_SUBJECTS,
        sessions=SMOKE_SESSIONS,
        trials_per_session=SMOKE_TRIALS_PER_SESSION,
        lazy=False,
    )
    print("Smoke dataset:", X_demo.shape, y_demo.shape)
    print("Subjects:", np.unique(subj_demo), "Class counts:", np.bincount(y_demo))
else:
    X_demo = y_demo = subj_demo = sess_demo = None



DATA MODULE
BCI IV-2a expected layout
  9 subjects × 2 sessions × 288 trials × 22 × 1000

                 ┌─────────────┐
                 │ subject A1  │
                 │ session 1/2 │
                 └──────┬──────┘
                        ...
                 ┌─────────────┐
                 │ subject A9  │
                 │ session 1/2 │
                 └──────┬──────┘
                        ▼
       X: (N, 22, 1000), y: (N,), subject: (N,)

Real .gdf:
  MNE Raw -> events -> cue at t=2 s -> [2,6) s -> 22×1000

Real .npz:
  expected X / y, optional subject / session keys

Real .mat:
  flexible recursive discovery of the first compatible arrays

Smoke dataset: (96, 22, 1000) (96,)
Subjects: [1 2] Class counts: [24 24 24 24]


In [4]:

# =====================================================================
# CELL 4 - PREPROCESSING: 5th-ORDER BUTTERWORTH + Z-SCORE + FILTER BANK
# =====================================================================

class EEGPreprocessor:
    """
    Paper preprocessing:
      NaN -> global sample mean
      5th-order Butterworth 1-38 Hz
      z-score using training statistics only
      10 sub-band decomposition
      4-s trial already expected as 22×1000
    """
    def __init__(
        self,
        fs: int = FS,
        broad_band: Tuple[float, float] = (1.0, 38.0),
        bands: Sequence[Tuple[float, float]] = BANDS,
        order: int = 5,
        eps: float = 1e-6,
        debug: bool = False,
    ):
        print_ascii(
            "PREPROCESSING MODULE",
            """
raw X (N,C,T)
   │
   ├── NaN → mean
   │
   ├── Butterworth 5th order, 1-38 Hz
   │
   ├── μ_train, σ_train
   │       X'=(X-μ_train)/(σ_train+ε)
   │
   └── 10 sub-bands
         1-4 | 4-8 | ... | 35-38 Hz
   → X_bands: (N,10,C,T)
""",
        )
        self.fs = fs
        self.broad_band = broad_band
        self.bands = list(bands)
        self.order = order
        self.eps = eps
        self.debug = debug

        self.mu_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

        self.broad_sos = signal.butter(
            order,
            broad_band,
            btype="bandpass",
            fs=fs,
            output="sos",
        )
        self.band_sos = {
            band: signal.butter(
                order,
                band,
                btype="bandpass",
                fs=fs,
                output="sos",
            )
            for band in self.bands
        }

    def _nan_fill(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float32).copy()
        bad = ~np.isfinite(X)
        if not bad.any():
            return X
        mean = np.nanmean(X)
        if not np.isfinite(mean):
            mean = 0.0
        X[bad] = mean
        return X

    def fit(self, X_train: np.ndarray):
        X = self._nan_fill(X_train)
        broad = signal.sosfiltfilt(self.broad_sos, X, axis=-1)
        # Training statistics only. Paper text labels the denominator
        # with σ²/variance; mathematically standard z-score uses std.
        self.mu_ = broad.mean(axis=(0, 2), keepdims=True)
        self.std_ = broad.std(axis=(0, 2), keepdims=True)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform_broad(self, X: np.ndarray) -> np.ndarray:
        if self.mu_ is None or self.std_ is None:
            raise RuntimeError("Call fit() before transform().")
        X = self._nan_fill(X)
        broad = signal.sosfiltfilt(self.broad_sos, X, axis=-1)
        return (broad - self.mu_) / self.std_

    def transform_filter_bank(self, X: np.ndarray) -> np.ndarray:
        Xz = self.transform_broad(X)
        out = []
        for band in self.bands:
            out.append(signal.sosfiltfilt(self.band_sos[band], Xz, axis=-1))
        fb = np.stack(out, axis=1)
        if self.debug:
            print("[Preprocessor] broad:", Xz.shape, "filter-bank:", fb.shape)
        return fb.astype(np.float32)

    def fit_transform(self, X_train: np.ndarray):
        self.fit(X_train)
        Xz = self.transform_broad(X_train)
        fb = self.transform_filter_bank(X_train)
        return Xz.astype(np.float32), fb

# Small sanity test
pre_demo = EEGPreprocessor(debug=True)
Xz_demo, fb_demo = pre_demo.fit_transform(X_demo)
print("Broad standardized:", Xz_demo.shape)
print("Filter-bank:", fb_demo.shape)



PREPROCESSING MODULE
raw X (N,C,T)
   │
   ├── NaN → mean
   │
   ├── Butterworth 5th order, 1-38 Hz
   │
   ├── μ_train, σ_train
   │       X'=(X-μ_train)/(σ_train+ε)
   │
   └── 10 sub-bands
         1-4 | 4-8 | ... | 35-38 Hz
   → X_bands: (N,10,C,T)

[Preprocessor] broad: (96, 22, 1000) filter-bank: (96, 10, 22, 1000)
Broad standardized: (96, 22, 1000)
Filter-bank: (96, 10, 22, 1000)


In [5]:

# =====================================================================
# CELL 5 - OVR FBCSP + LASSO FEATURE SELECTION
# =====================================================================

class OVRFBCSPLasso:
    """
    OVR FBCSP + LASSO.

    Candidate dimensions:
        10 bands × 4 OVR class bifurcations × 4 eigenvectors = 160.
    """

    def __init__(
        self,
        n_classes: int = N_CLASSES,
        m: int = CSP_M,
        random_state: int = SEED,
        debug: bool = False,
    ):
        print_ascii(
            "FBCSP + LASSO MODULE",
            """
10 filtered sub-bands
       │
       ▼
For each band:
  class 0 vs rest → 4 CSP filters
  class 1 vs rest → 4 CSP filters
  class 2 vs rest → 4 CSP filters
  class 3 vs rest → 4 CSP filters
       │
       ▼
160 candidate log-variance features
       │
       ▼
LASSO(L1)
       │
       ├────────────► selected feature indices
       │
       └────────────► sparse W_csp
                      dynamic Var
""",
        )
        self.n_classes = n_classes
        self.m = m
        self.random_state = random_state
        self.debug = debug

        self.filters_: Optional[np.ndarray] = None
        self.meta_: List[Dict] = []
        self.selected_idx_: Optional[np.ndarray] = None
        self.lasso_model_: Optional[LassoCV] = None
        self.feature_scaler_: Optional[StandardScaler] = None

    @staticmethod
    def _normalized_covariance(X: np.ndarray, eps=1e-12) -> np.ndarray:
        cov = X @ X.T
        tr = np.trace(cov)
        return cov / max(tr, eps)

    def _class_covariance(self, X_band: np.ndarray, labels: np.ndarray, cls: int):
        idx = labels == cls
        if idx.sum() < 2:
            raise ValueError(f"Not enough trials for class {cls}.")
        covs = np.stack(
            [self._normalized_covariance(x) for x in X_band[idx]],
            axis=0,
        )
        return covs.mean(axis=0)

    def fit(self, X_fb: np.ndarray, y: np.ndarray):
        # X_fb: (N,10,C,T)
        n, n_bands, channels, _ = X_fb.shape
        filters = []
        meta = []

        for b in range(n_bands):
            Xb = X_fb[:, b]
            for cls in range(self.n_classes):
                Rc = self._class_covariance(Xb, y, cls)
                rest = Xb[y != cls]
                Rr = np.stack(
                    [self._normalized_covariance(x) for x in rest],
                    axis=0,
                ).mean(axis=0)

                # Numerical regularization for generalized eigensolver.
                reg = 1e-6 * np.eye(channels)
                vals, vecs = linalg.eigh(Rc + reg, Rr + reg)

                order = np.argsort(vals)[::-1][: self.m]
                W = vecs[:, order]
                for j in range(self.m):
                    filters.append(W[:, j])
                    meta.append(
                        {"band_index": b, "class": cls, "eigen_index": j}
                    )

        # Columns: C × 160
        W_all = np.stack(filters, axis=1).astype(np.float64)

        # Log-variance feature matrix N × 160.
        F_all = []
        for i in range(n):
            sample_features = []
            for k, info in enumerate(meta):
                band_idx = info["band_index"]
                proj = W_all[:, k].T @ X_fb[i, band_idx]
                var = np.var(proj) + 1e-12
                sample_features.append(np.log(var))
            F_all.append(sample_features)
        F_all = np.asarray(F_all, dtype=np.float64)

        scaler = StandardScaler()
        F_scaled = scaler.fit_transform(F_all)
        y_float = y.astype(np.float64)

        # Paper uses scalar LASSO regression. We reproduce that formulation.
        lasso = LassoCV(
            cv=min(5, max(2, np.min(np.bincount(y)))),
            random_state=self.random_state,
            n_jobs=None,
            max_iter=10000,
        )
        lasso.fit(F_scaled, y_float)
        coef = np.abs(lasso.coef_)

        selected = np.flatnonzero(coef > LASSO_EPS)

        # Robust fallback for synthetic / tiny smoke data when CV drives
        # every coefficient to zero. This is not a paper change in full mode;
        # it only prevents a degenerate smoke-test run.
        if len(selected) == 0:
            corr = np.abs(np.corrcoef(F_scaled, y_float, rowvar=False)[-1, :-1])
            selected = np.argsort(np.nan_to_num(corr, nan=0.0))[-max(4, self.m):]
            warnings.warn(
                "LASSO selected zero coefficients; smoke-test fallback "
                "selected top correlated CSP features."
            )

        selected = np.sort(selected)

        self.filters_ = W_all
        self.meta_ = meta
        self.selected_idx_ = selected
        self.lasso_model_ = lasso
        self.feature_scaler_ = scaler

        if self.debug:
            print("Candidate feature matrix:", F_all.shape)
            print("Selected features:", len(selected))
            print("Selected meta:", [meta[i] for i in selected[:10]])

        return self

    def transform_features(self, X_fb: np.ndarray) -> np.ndarray:
        if self.filters_ is None or self.selected_idx_ is None:
            raise RuntimeError("Call fit() first.")
        selected = self.selected_idx_
        out = []
        for i in range(len(X_fb)):
            feats = []
            for k in selected:
                info = self.meta_[k]
                proj = self.filters_[:, k].T @ X_fb[i, info["band_index"]]
                feats.append(np.log(np.var(proj) + 1e-12))
            out.append(feats)
        return np.asarray(out, dtype=np.float32)

    def transform_sparse_fb(self, X_fb: np.ndarray) -> np.ndarray:
        """
        Apply the retained spatial filters to their associated sub-band EEG.

        Output:
            Z = (N, Var, T), then add singleton channel dim for Dψ:
            (N,1,Var,T)
        """
        if self.filters_ is None or self.selected_idx_ is None:
            raise RuntimeError("Call fit() first.")
        selected = self.selected_idx_
        projected = []
        for k in selected:
            info = self.meta_[k]
            band = X_fb[:, info["band_index"]]  # N,C,T
            Wk = self.filters_[:, k]            # C
            projected.append(np.einsum("c,nct->nt", Wk, band))
        Z = np.stack(projected, axis=1)
        return Z.astype(np.float32)[:, None, :, :]

    @property
    def var_(self) -> int:
        return 0 if self.selected_idx_ is None else len(self.selected_idx_)

fbcsp_demo = OVRFBCSPLasso(debug=True)
fbcsp_demo.fit(fb_demo, y_demo)
Z_demo = fbcsp_demo.transform_sparse_fb(fb_demo)
print("Sparse FB discriminator input:", Z_demo.shape)



FBCSP + LASSO MODULE
10 filtered sub-bands
       │
       ▼
For each band:
  class 0 vs rest → 4 CSP filters
  class 1 vs rest → 4 CSP filters
  class 2 vs rest → 4 CSP filters
  class 3 vs rest → 4 CSP filters
       │
       ▼
160 candidate log-variance features
       │
       ▼
LASSO(L1)
       │
       ├────────────► selected feature indices
       │
       └────────────► sparse W_csp
                      dynamic Var

Candidate feature matrix: (96, 160)
Selected features: 42
Selected meta: [{'band_index': 0, 'class': 1, 'eigen_index': 0}, {'band_index': 0, 'class': 3, 'eigen_index': 2}, {'band_index': 1, 'class': 1, 'eigen_index': 3}, {'band_index': 1, 'class': 3, 'eigen_index': 0}, {'band_index': 1, 'class': 3, 'eigen_index': 2}, {'band_index': 2, 'class': 0, 'eigen_index': 0}, {'band_index': 2, 'class': 0, 'eigen_index': 1}, {'band_index': 2, 'class': 0, 'eigen_index': 2}, {'band_index': 2, 'class': 1, 'eigen_index': 0}, {'band_index': 2, 'class': 1, 'eigen_index': 1}]
Sparse

In [6]:

# =====================================================================
# CELL 6 - FBGAN: GENERATOR + Dφ + Dψ + ADVERSARIAL TRAINING
# =====================================================================

class FBGANGenerator(nn.Module):
    def __init__(self, latent_dim=GAN_LATENT_DIM, debug=False):
        super().__init__()
        print_ascii(
            "FBGAN GENERATOR",
            """
z ~ N(0,I), shape (B,1600)
          │
          ▼
     FC 1600→256000
          │
          ▼
   reshape 128×20×100
          │
          ▼
ConvTrans1 128→128, k=(3,15), s=(1,3)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans2 128→128, k=(3,15), s=(1,3)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans3 128→64,  k=(3,5),  s=(1,2)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans4 64→32,   k=(4,5),  s=(2,1)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans5 32→1,    k=(1,2),  s=(1,1), Tanh
          │
          ▼
non-learnable shape adapter
          │
          ▼
(B,1,22,1000)
""",
        )
        self.tracker = ShapeTracker(debug, "Gθ")
        self.fc = nn.Linear(latent_dim, 256000)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 128, kernel_size=(3, 15), stride=(1, 3)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(128, 128, kernel_size=(3, 15), stride=(1, 3)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=(3, 5), stride=(1, 2)),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=(4, 5), stride=(2, 1)),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(32, 1, kernel_size=(1, 2), stride=(1, 1)),
            nn.Tanh(),
        )

    def forward(self, z):
        self.tracker("z", z)
        x = self.fc(z).view(z.size(0), 128, 20, 100)
        self.tracker("after FC reshape", x)
        x = self.net(x)
        self.tracker("raw ConvTrans5", x)
        # The published Table 1 does not specify padding/reshape details.
        # Interpolation is therefore used only as a deterministic shape
        # adapter to enforce the stated EEG output size.
        x = F.interpolate(
            x,
            size=(N_CHANNELS, N_TIME),
            mode="bilinear",
            align_corners=False,
        )
        self.tracker("EEG output", x)
        return x


class _DiscBase(nn.Module):
    def _shape_to_750(self, x):
        # Published FC input = 750. Literal valid convolution geometry
        # from the table does not yield 750; adaptive pooling is a
        # non-trainable geometry adapter that preserves the FC=750 spec.
        return F.adaptive_avg_pool2d(x, (1, 25))


class RawEEGDiscriminator(_DiscBase):
    def __init__(self, debug=False):
        super().__init__()
        print_ascii(
            "FBGAN RAW EEG DISCRIMINATOR Dφ",
            """
(B,1,22,1000)
   │
   ├─ Conv1 1→10,  k=(1,23)
   ├─ Conv2 10→30, k=(22,1)
   ├─ Conv3 30→30, k=(1,17)
   ├─ MaxPool (1,6), stride (1,6)
   ├─ Conv4 30→30, k=(1,7), stride (1,7)
   ├─ MaxPool (1,6), stride (1,6)
   ├─ adaptive pool → (1,25)
   └─ FC 750→1
                    │
                    ▼
                 realism
""",
        )
        self.tracker = ShapeTracker(debug, "Dφ")
        self.features = nn.Sequential(
            nn.Conv2d(1, 10, kernel_size=(1, 23)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(10, 30, kernel_size=(22, 1)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(30, 30, kernel_size=(1, 17)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6)),

            nn.Conv2d(30, 30, kernel_size=(1, 7), stride=(1, 7)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6)),
        )
        self.fc = nn.Linear(750, 1)

    def forward(self, x):
        self.tracker("input", x)
        x = self.features(x)
        self.tracker("conv features", x)
        x = self._shape_to_750(x)
        self.tracker("750 adapter", x)
        return self.fc(x.flatten(1))


class SparseFBDiscriminator(_DiscBase):
    def __init__(self, var: int, debug=False):
        super().__init__()
        if var < 4:
            raise ValueError("Dψ needs at least 4 sparse spatial dimensions.")

        # After Conv2 with k=4, stride=4, the sparse height is reduced.
        conv3_h = max(1, (var - 4) // 4 + 1)

        print_ascii(
            "FBGAN SPARSE FILTER-BANK DISCRIMINATOR Dψ",
            f"""
(B,1,Var={var},1000)
   │
   ├─ Conv1 1→10,  k=(1,23)
   ├─ Conv2 10→30, k=(4,1), stride=(4,1)
   ├─ Conv3 30→30, k=({conv3_h},1)  [Var-dependent]
   ├─ Conv4 30→30, k=(1,17)
   ├─ MaxPool (1,6), stride (1,6)
   ├─ Conv5 30→30, k=(1,7)
   ├─ MaxPool (1,6), stride (1,6)
   ├─ adaptive pool → (1,25)
   └─ FC 750→1
""",
        )
        self.tracker = ShapeTracker(debug, "Dψ")
        self.features = nn.Sequential(
            nn.Conv2d(1, 10, kernel_size=(1, 23)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(10, 30, kernel_size=(4, 1), stride=(4, 1)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(30, 30, kernel_size=(conv3_h, 1)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(30, 30, kernel_size=(1, 17)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6)),

            nn.Conv2d(30, 30, kernel_size=(1, 7)),
            nn.LeakyReLU(0.2, inplace=True),

            nn.MaxPool2d(kernel_size=(1, 6), stride=(1, 6)),
        )
        self.fc = nn.Linear(750, 1)
        self.var = var

    def forward(self, x):
        self.tracker("input", x)
        x = self.features(x)
        self.tracker("conv features", x)
        x = self._shape_to_750(x)
        self.tracker("750 adapter", x)
        return self.fc(x.flatten(1))


class GANSignalScaler:
    """Practical bounded adapter because the generator ends with Tanh."""
    def __init__(self, quantile=0.995, eps=1e-6):
        self.scale = None
        self.quantile = quantile
        self.eps = eps

    def fit(self, X):
        q = np.quantile(np.abs(X), self.quantile)
        self.scale = max(float(q), self.eps)
        return self

    def transform(self, X):
        if self.scale is None:
            raise RuntimeError("Fit scaler first.")
        return np.clip(X / self.scale, -1.0, 1.0)

    def inverse(self, X):
        if self.scale is None:
            raise RuntimeError("Fit scaler first.")
        return X * self.scale


class TorchSparseFBProjector(nn.Module):
    """
    Differentiable approximation of Equation (6):

        Z = W_csp^T X'_band

    For each retained sparse CSP filter, a fixed FFT-domain band-pass mask
    is applied and the signal is projected by the saved spatial filter.

    This keeps the Dψ gradient path differentiable back to Gθ.
    """
    def __init__(
        self,
        filters: np.ndarray,
        meta: List[Dict],
        selected_idx: np.ndarray,
        bands: Sequence[Tuple[float, float]],
        fs: int = FS,
    ):
        super().__init__()
        print_ascii(
            "DIFFERENTIABLE SPARSE-FB PROJECTOR",
            """
Generated EEG X'
      │
      ├─ fixed FFT band mask for each retained band
      │
      ├─ inverse FFT → band-limited EEG
      │
      └─ spatial projection with saved sparse CSP w_k
                    │
                    ▼
          Z = (B,1,Var,T)
                    │
                    ▼
                   Dψ
""",
        )
        selected = np.asarray(selected_idx, dtype=int)
        weights = torch.as_tensor(
            filters[:, selected].T,
            dtype=torch.float32,
        )
        band_edges = [bands[meta[int(k)]["band_index"]] for k in selected]

        freqs = np.fft.rfftfreq(N_TIME, d=1.0 / fs)
        masks = []
        for lo, hi in band_edges:
            mask = ((freqs >= lo) & (freqs < hi)).astype(np.float32)
            masks.append(mask)
        masks = torch.as_tensor(np.stack(masks), dtype=torch.float32)

        self.register_buffer("weights", weights)  # Var × C
        self.register_buffer("masks", masks)      # Var × Freq

    def forward(self, x):
        # x: B×1×C×T
        x = x[:, 0]
        xf = torch.fft.rfft(x, dim=-1)
        # Shared spectral transform, one fixed band mask per selected filter.
        # Result: B×Var×C×Freq.
        band = xf[:, None, :, :] * self.masks[None, :, None, :]
        band_t = torch.fft.irfft(
            band,
            n=x.shape[-1],
            dim=-1,
        )
        # w_k^T X_k for each retained CSP filter.
        z = torch.einsum("vc,bvct->bvt", self.weights, band_t)
        return z[:, None]


class FBGAN:
    def __init__(
        self,
        projector: TorchSparseFBProjector,
        debug=False,
    ):
        print_ascii(
            "FBGAN DUAL-DISCRIMINATOR TRAINING",
            """
real EEG ───────────────► Dφ ─────┐
                                  │
fake EEG ─► G(z) ─────────► Dφ ───┤
                                  │
real EEG ─► sparse W_csp/FB ► Dψ ─┤── adversarial objective
fake EEG ─► sparse W_csp/FB ► Dψ ─┘
                                  │
                    Gθ learns to fool BOTH Dφ and Dψ
""",
        )
        self.G = FBGANGenerator(debug=debug).to(DEVICE)
        self.D_phi = RawEEGDiscriminator(debug=debug).to(DEVICE)
        self.D_psi = SparseFBDiscriminator(
            var=int(projector.weights.shape[0]),
            debug=debug,
        ).to(DEVICE)

        self.projector = projector.to(DEVICE)
        self.g_opt = torch.optim.Adam(self.G.parameters(), lr=GAN_LR)
        self.dphi_opt = torch.optim.Adam(self.D_phi.parameters(), lr=GAN_LR)
        self.dpsi_opt = torch.optim.Adam(self.D_psi.parameters(), lr=GAN_LR)
        self.bce = nn.BCEWithLogitsLoss()

    def train_from_target_class(
        self,
        real_x: np.ndarray,
        epochs: int = 1,
        steps_per_epoch: Optional[int] = None,
    ):
        """
        real_x:
            target-class EEG in broad-band standardized space,
            shape (N,22,1000).
        """
        scaler = GANSignalScaler().fit(real_x)
        real_scaled = scaler.transform(real_x).astype(np.float32)

        loader = DataLoader(
            TensorDataset(torch.from_numpy(real_scaled[:, None])),
            batch_size=GAN_BATCH_SIZE,
            shuffle=True,
            drop_last=False,
        )

        history = []
        for epoch in range(epochs):
            for step, (real_x_b,) in enumerate(loader):
                if steps_per_epoch is not None and step >= steps_per_epoch:
                    break

                real_x_b = real_x_b.to(DEVICE)

                # ========================= D_phi ==========================
                z = torch.randn(len(real_x_b), GAN_LATENT_DIM, device=DEVICE)
                with torch.no_grad():
                    fake_x = self.G(z)

                dphi_real = self.D_phi(real_x_b)
                dphi_fake = self.D_phi(fake_x)
                loss_dphi = (
                    self.bce(dphi_real, torch.ones_like(dphi_real))
                    + self.bce(dphi_fake, torch.zeros_like(dphi_fake))
                )

                self.dphi_opt.zero_grad(set_to_none=True)
                loss_dphi.backward()
                self.dphi_opt.step()

                # ========================= D_psi ==========================
                real_z = self.projector(real_x_b)
                fake_z = self.projector(fake_x.detach())

                dpsi_real = self.D_psi(real_z)
                dpsi_fake = self.D_psi(fake_z)
                loss_dpsi = (
                    self.bce(dpsi_real, torch.ones_like(dpsi_real))
                    + self.bce(dpsi_fake, torch.zeros_like(dpsi_fake))
                )

                self.dpsi_opt.zero_grad(set_to_none=True)
                loss_dpsi.backward()
                self.dpsi_opt.step()

                # ============================= G ==========================
                z = torch.randn(len(real_x_b), GAN_LATENT_DIM, device=DEVICE)
                gen_x = self.G(z)
                gen_z = self.projector(gen_x)

                g_loss = (
                    self.bce(
                        self.D_phi(gen_x),
                        torch.ones_like(dphi_real),
                    )
                    + self.bce(
                        self.D_psi(gen_z),
                        torch.ones_like(dpsi_real),
                    )
                )

                self.g_opt.zero_grad(set_to_none=True)
                g_loss.backward()
                self.g_opt.step()

                history.append(
                    {
                        "epoch": epoch + 1,
                        "step": step + 1,
                        "dphi": float(loss_dphi.item()),
                        "dpsi": float(loss_dpsi.item()),
                        "g": float(g_loss.item()),
                    }
                )
        return history, scaler

    @torch.no_grad()
    def generate(self, n: int, scaler: Optional[GANSignalScaler] = None):
        self.G.eval()
        outs = []
        for start in range(0, n, GAN_BATCH_SIZE):
            b = min(GAN_BATCH_SIZE, n - start)
            z = torch.randn(b, GAN_LATENT_DIM, device=DEVICE)
            fake = self.G(z).cpu().numpy()[:, 0]
            if scaler is not None:
                fake = scaler.inverse(fake)
            outs.append(fake.astype(np.float32))
        return np.concatenate(outs, axis=0)


# ---------------------- Architecture-only verification -----------------
# Uses a tiny dummy sparse projection so Cell 6 is runnable on its own.
dummy_filters = np.zeros((N_CHANNELS, 4), dtype=np.float32)
dummy_filters[:4, :4] = np.eye(4, dtype=np.float32)
dummy_meta = [
    {"band_index": 0, "class": 0, "eigen_index": i} for i in range(4)
]
dummy_selected = np.arange(4)
dummy_projector = TorchSparseFBProjector(
    dummy_filters,
    dummy_meta,
    dummy_selected,
    BANDS,
).to(DEVICE)

G_demo = FBGANGenerator(debug=True).to(DEVICE)
z_demo = torch.randn(2, GAN_LATENT_DIM, device=DEVICE)
with torch.no_grad():
    fake_demo = G_demo(z_demo)
print("Generator verified:", tuple(fake_demo.shape))

Dphi_demo = RawEEGDiscriminator(debug=True).to(DEVICE)
with torch.no_grad():
    dphi_score = Dphi_demo(fake_demo)
print("D_phi output:", tuple(dphi_score.shape))

Dpsi_demo = SparseFBDiscriminator(4, debug=True).to(DEVICE)
with torch.no_grad():
    dummy_z = dummy_projector(fake_demo)
    dpsi_score = Dpsi_demo(dummy_z)
print("D_psi output:", tuple(dpsi_score.shape))



DIFFERENTIABLE SPARSE-FB PROJECTOR
Generated EEG X'
      │
      ├─ fixed FFT band mask for each retained band
      │
      ├─ inverse FFT → band-limited EEG
      │
      └─ spatial projection with saved sparse CSP w_k
                    │
                    ▼
          Z = (B,1,Var,T)
                    │
                    ▼
                   Dψ


FBGAN GENERATOR
z ~ N(0,I), shape (B,1600)
          │
          ▼
     FC 1600→256000
          │
          ▼
   reshape 128×20×100
          │
          ▼
ConvTrans1 128→128, k=(3,15), s=(1,3)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans2 128→128, k=(3,15), s=(1,3)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans3 128→64,  k=(3,5),  s=(1,2)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans4 64→32,   k=(4,5),  s=(2,1)
          │ BatchNorm + LeakyReLU
          ▼
ConvTrans5 32→1,    k=(1,2),  s=(1,1), Tanh
          │
          ▼
non-learnable shape adapter
          │
          ▼
(B,1,22,1000)

[ShapeTrack

RuntimeError: Adaptive pool MPS: input sizes must be divisible by output sizes. Non-divisible input sizes are not implemented on MPS device yet. For now, you can manually transfer tensor to cpu in this case. Please refer to [this issue](https://github.com/pytorch/pytorch/issues/96056)

In [ ]:

# =====================================================================
# CELL 7 - CRNN-DF: SPATIAL CNN + 2-LAYER LSTM + FEATURE EXTRACTOR
# =====================================================================

class CRNNDF(nn.Module):
    """
    CRNN-DF classifier.

    The paper specifies:
      Conv2D kernel = C×45, stride=1, BatchNorm, ReLU, Dropout=0.5
      MaxPool kernel = 1×75, stride=10
      2× LSTM hidden=64, dropout=0.5
    The figure does not state the Conv2D output-channel count, so this
    implementation exposes it as a parameter (default 40).
    """
    def __init__(self, n_classes=N_CLASSES, conv_channels=40, debug=False):
        super().__init__()
        print_ascii(
            "CRNN-DF CLASSIFIER",
            f"""
Input X: (B,1,22,1000)
     │
     ▼
Conv2D: out={conv_channels}, kernel=(22,45), stride=1
     │ BatchNorm + ReLU + Dropout(0.5)
     ▼
(B,{conv_channels},1,956)
     │
     ▼
MaxPool2D kernel=(1,75), stride=(1,10)
     │
     ▼
(B,{conv_channels},1,89)
     │
     ▼
reshape to sequence (B,89,{conv_channels})
     │
     ▼
LSTM-1 hidden=64, dropout=0.5
     │
     ▼
LSTM-2 hidden=64, dropout=0.5
     │
     ▼
discriminative feature vector v ∈ R^64
     │
     ▼
FC 64→4
"""
        )
        self.tracker = ShapeTracker(debug, "CRNN-DF")
        self.spatial = nn.Sequential(
            nn.Conv2d(
                1,
                conv_channels,
                kernel_size=(N_CHANNELS, 45),
                stride=(1, 1),
                bias=False,
            ),
            nn.BatchNorm2d(conv_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.MaxPool2d(kernel_size=(1, 75), stride=(1, 10)),
        )
        self.lstm1 = nn.LSTM(
            input_size=conv_channels,
            hidden_size=64,
            batch_first=True,
        )
        self.drop1 = nn.Dropout(0.5)
        self.lstm2 = nn.LSTM(
            input_size=64,
            hidden_size=64,
            batch_first=True,
        )
        self.drop2 = nn.Dropout(0.5)
        self.classifier = nn.Linear(64, n_classes)

    def extract_features(self, x):
        self.tracker("input", x)
        if x.ndim == 3:
            x = x.unsqueeze(1)
        x = self.spatial(x)
        self.tracker("CNN output", x)
        x = x.squeeze(2).transpose(1, 2)
        self.tracker("LSTM sequence", x)
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        self.tracker("LSTM1", x)
        x, _ = self.lstm2(x)
        x = self.drop2(x)
        self.tracker("LSTM2", x)
        v = x[:, -1, :]
        self.tracker("feature v", v)
        return v

    def forward(self, x):
        v = self.extract_features(x)
        logits = self.classifier(v)
        self.tracker("logits", logits)
        return logits, v

crnn_demo = CRNNDF(debug=True).to(DEVICE)
x_crnn_demo = torch.from_numpy(X_demo[:4]).float().to(DEVICE)
with torch.no_grad():
    logits_demo, feat_demo = crnn_demo(x_crnn_demo)
print("CRNN logits:", logits_demo.shape)
print("CRNN feature:", feat_demo.shape)
print("Trainable parameters:", count_parameters(crnn_demo))


In [ ]:

# =====================================================================
# CELL 8 - CENTRAL DISTANCE LOSS + CENTROID INITIALIZATION/REPULSION
# =====================================================================

class DiscriminativeFeatureController:
    """
    Implements the paper's central distance loss and center-shift rule.

    Note: the source paper initializes centroids from all training samples
    before classifier training, then shifts them every 15 epochs by alpha=0.02.
    """
    def __init__(
        self,
        n_classes=N_CLASSES,
        feature_dim=64,
        lambda_center=CENTER_LAMBDA,
        alpha=CENTER_SHIFT_ALPHA,
        update_every=CENTER_SHIFT_EVERY,
        device=DEVICE,
        debug=False,
    ):
        print_ascii(
            "DISCRIMINATIVE FEATURE / CENTER MODULE",
            f"""
feature vectors v_i ∈ R^{feature_dim}
        │
        ├── class-wise centroid cen_j^k
        │
        ├── central distance loss
        │       Lcen = 1/b Σ ||v_i-cen_y||²
        │
        └── every {update_every} epochs:
              vc = mean_j(cen_j)
              cen_j ← cen_j + {alpha}
                       * (cen_j-vc)/||cen_j-vc||
        │
        ▼
L = CrossEntropy + {lambda_center} × Lcen
"""
        )
        self.n_classes = n_classes
        self.feature_dim = feature_dim
        self.lambda_center = lambda_center
        self.alpha = alpha
        self.update_every = update_every
        self.device = device
        self.debug = debug
        self.centroids = torch.zeros(
            n_classes, feature_dim, device=device, dtype=torch.float32
        )

    @torch.no_grad()
    def initialize(self, model: nn.Module, loader: DataLoader):
        model.eval()
        sums = torch.zeros_like(self.centroids)
        counts = torch.zeros(self.n_classes, device=self.device)

        for xb, yb in loader:
            xb = xb.to(self.device)
            yb = yb.to(self.device)
            _, feats = model(xb)
            for cls in range(self.n_classes):
                idx = yb == cls
                if idx.any():
                    sums[cls] += feats[idx].sum(dim=0)
                    counts[cls] += idx.sum()

        for cls in range(self.n_classes):
            if counts[cls] > 0:
                self.centroids[cls] = sums[cls] / counts[cls]

        model.train()

    def center_loss(self, features, labels):
        c = self.centroids[labels]
        return ((features - c) ** 2).sum(dim=1).mean()

    @torch.no_grad()
    def shift_centroids(self):
        center = self.centroids.mean(dim=0, keepdim=True)
        directions = self.centroids - center
        norms = torch.norm(directions, dim=1, keepdim=True).clamp_min(1e-8)
        self.centroids += self.alpha * directions / norms

        if self.debug:
            pair_dist = torch.cdist(self.centroids, self.centroids)
            print(
                "[Centroid shift] min off-diagonal distance:",
                float(pair_dist[pair_dist > 0].min().item()),
            )

def train_crnn_df(
    model: CRNNDF,
    X_train: np.ndarray,
    y_train: np.ndarray,
    epochs: int = 2,
    lambda_center: float = CENTER_LAMBDA,
    debug: bool = False,
):
    ds = TensorDataset(
        torch.from_numpy(X_train).float(),
        torch.from_numpy(y_train).long(),
    )
    loader = DataLoader(ds, batch_size=CLS_BATCH_SIZE, shuffle=True)

    controller = DiscriminativeFeatureController(
        n_classes=N_CLASSES,
        feature_dim=64,
        lambda_center=lambda_center,
        debug=debug,
    )
    controller.initialize(model, DataLoader(ds, batch_size=CLS_BATCH_SIZE, shuffle=False))

    optimizer = torch.optim.Adam(model.parameters(), lr=CLS_LR)
    ce = nn.CrossEntropyLoss()

    history = []
    model.train()

    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        running_acc = 0.0
        n_seen = 0

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits, feats = model(xb)
            loss_ce = ce(logits, yb)
            loss_cen = controller.center_loss(feats, yb)
            loss = loss_ce + lambda_center * loss_cen

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            pred = logits.argmax(dim=1)
            n = len(yb)
            running_loss += float(loss.item()) * n
            running_acc += float((pred == yb).float().mean().item()) * n
            n_seen += n

        if epoch % controller.update_every == 0:
            controller.shift_centroids()

        history.append({
            "epoch": epoch,
            "loss": running_loss / max(n_seen, 1),
            "train_acc": running_acc / max(n_seen, 1),
        })
        if debug:
            print(history[-1])

    return model, controller, history



In [ ]:

# =====================================================================
# CELL 9 - LOSO TRAINER + TARGET-SUBJECT FBGAN AUGMENTATION
# =====================================================================

print_ascii(
    "LOSO TRAINING PIPELINE",
    r"""
                    9 subjects
                       │
          ┌────────────┴────────────┐
          │                         │
   source subjects (8)       target subject (1)
          │                         │
          │                   preprocessing
          │                         │
          │                   target FBCSP
          │                         │
          │                    + LASSO W
          │                         │
          │                    4 class FBGANs
          │                         │
          │                     fake EEG
          │                         │
          └──────────────┬──────────┘
                         ▼
               augmented source train
                         │
                         ▼
                      CRNN-DF
                         │
                         ▼
                 untouched target test
                         │
                         ▼
               accuracy / confusion / t-SNE

Paper LOSO:
  8 subjects × 2 sessions × 288 = 4608 train trials
  target subject = 576 test trials
Paper augmentation:
  3000 fake trials = 750 per class
"""
)

def split_loso(
    X: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    target_subject: int,
):
    train_mask = subjects != target_subject
    test_mask = subjects == target_subject
    return (
        X[train_mask],
        y[train_mask],
        X[test_mask],
        y[test_mask],
    )

def train_target_fbgans(
    X_target: np.ndarray,
    y_target: np.ndarray,
    preprocessor: EEGPreprocessor,
    fbcsp: OVRFBCSPLasso,
    n_per_class: int,
    gan_epochs: int,
    gan_steps: Optional[int],
):
    """
    Train one FBGAN per target class, matching the source's parallel
    category-specific FBGAN setup.
    """
    fb_target = preprocessor.transform_filter_bank(X_target)
    sparse_target = fbcsp.transform_sparse_fb(fb_target)

    generated = []
    gan_histories = []

    for cls in range(N_CLASSES):
        print(f"\n[FBGAN] target class {cls}: {CLASS_NAMES[cls]}")
        idx = np.where(y_target == cls)[0]
        if len(idx) < 4:
            warnings.warn(f"Class {cls} has only {len(idx)} target samples.")
            continue

        raw_class = X_target[idx]
        gan = FBGAN(
            projector=projector,
            debug=False,
        )
        hist, scaler = gan.train_from_target_class(
            real_x=raw_class,
            epochs=gan_epochs,
            steps_per_epoch=gan_steps,
        )
        gan_histories.append(hist)

        fake = gan.generate(
            n=n_per_class,
            scaler=scaler,
        )
        generated.append(fake)

    if not generated:
        return (
            np.empty((0, N_CHANNELS, N_TIME), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
            gan_histories,
        )

    X_fake = np.concatenate(generated, axis=0)
    y_fake = np.concatenate(
        [
            np.full(n_per_class, cls, dtype=np.int64)
            for cls in range(len(generated))
        ]
    )
    return X_fake, y_fake, gan_histories

def run_loso_experiment(
    X: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    max_subjects: Optional[int] = None,
    use_fbgAN: bool = True,
    n_aug: int = DEFAULT_N_AUG,
    gan_epochs: int = SMOKE_GAN_EPOCHS,
    gan_steps: Optional[int] = SMOKE_GAN_STEPS_PER_EPOCH,
    classifier_epochs: int = SMOKE_CLS_EPOCHS,
):
    results = []
    all_predictions = []
    all_features = []

    target_subjects = sorted(np.unique(subjects))
    if max_subjects is not None:
        target_subjects = target_subjects[:max_subjects]

    for target in target_subjects:
        print("\n" + "#" * 78)
        print(f"LOSO TARGET SUBJECT A{target}")
        print("#" * 78)

        Xtr, ytr, Xte, yte = split_loso(X, y, subjects, target)

        # Fit preprocessing on source-subject training only.
        pre = EEGPreprocessor(debug=False)
        pre.fit(Xtr)

        Xtr_z = pre.transform_broad(Xtr)
        Xte_z = pre.transform_broad(Xte)
        Xtr_fb = pre.transform_filter_bank(Xtr)

        # Target-specific FBCSP/LASSO is used as the FBGAN spatial constraint.
        # This follows the target-aware augmentation strategy in the paper.
        Xtarget_fb = pre.transform_filter_bank(Xte)
        fbcsp = OVRFBCSPLasso(debug=False)
        fbcsp.fit(Xtarget_fb, yte)

        X_fake = np.empty((0, N_CHANNELS, N_TIME), dtype=np.float32)
        y_fake = np.empty((0,), dtype=np.int64)
        gan_hist = []

        if use_fbgAN and n_aug > 0:
            per_class = max(1, n_aug // N_CLASSES)
            X_fake, y_fake, gan_hist = train_target_fbgans(
                X_target=Xte_z,
                y_target=yte,
                preprocessor=pre,
                fbcsp=fbcsp,
                n_per_class=per_class,
                gan_epochs=gan_epochs,
                gan_steps=gan_steps,
            )

        # CRNN training uses source training data plus FBGAN fake target data.
        X_aug = np.concatenate([Xtr_z, X_fake], axis=0)
        y_aug = np.concatenate([ytr, y_fake], axis=0)

        model = CRNNDF(debug=False).to(DEVICE)
        model, center_ctrl, history = train_crnn_df(
            model,
            X_aug.astype(np.float32),
            y_aug.astype(np.int64),
            epochs=classifier_epochs,
            lambda_center=CENTER_LAMBDA,
            debug=False,
        )

        # Evaluate on real target subject only.
        model.eval()
        with torch.no_grad():
            x_test_t = torch.from_numpy(Xte_z).float().to(DEVICE)
            logits, feats = model(x_test_t)
            pred = logits.argmax(dim=1).cpu().numpy()
            feats = feats.cpu().numpy()

        acc = accuracy_score(yte, pred)
        print(f"Target A{target}: accuracy={100*acc:.2f}%")

        results.append({
            "subject": f"A{target}",
            "n_train_real": len(Xtr),
            "n_train_fake": len(X_fake),
            "accuracy": acc,
            "gan_history_steps": sum(len(h) for h in gan_hist),
        })
        all_predictions.append(
            {
                "subject": target,
                "y_true": yte.copy(),
                "y_pred": pred.copy(),
            }
        )
        all_features.append(
            {
                "subject": target,
                "features": feats,
                "labels": yte.copy(),
            }
        )

    results_df = None
    try:
        import pandas as pd
        results_df = pd.DataFrame(results)
    except Exception:
        results_df = results

    return results_df, all_predictions, all_features

# ------------------------- Run notebook smoke --------------------------

if RUN_SMOKE_EXPERIMENT:
    smoke_results, smoke_predictions, smoke_features = run_loso_experiment(
        X_demo,
        y_demo,
        subj_demo,
        max_subjects=min(SMOKE_SUBJECTS, len(np.unique(subj_demo))),
        use_fbgAN=True,
        n_aug=DEFAULT_N_AUG,
        gan_epochs=SMOKE_GAN_EPOCHS,
        gan_steps=SMOKE_GAN_STEPS_PER_EPOCH,
        classifier_epochs=SMOKE_CLS_EPOCHS,
    )
    print("\nSmoke LOSO results:")
    print(smoke_results)

# Full paper-style run:
# NOTE: this is intentionally not executed by default.
#
# full_results, full_predictions, full_features = run_loso_experiment(
#     X_full, y_full, subjects_full,
#     max_subjects=9,
#     use_fbgAN=True,
#     n_aug=3000,               # 750/class
#     gan_epochs=FULL_GAN_EPOCHS,
#     gan_steps=None,
#     classifier_epochs=FULL_CLS_EPOCHS,
# )


In [ ]:

# =====================================================================
# CELL 10 - PERFORMANCE EVALUATION, CONFUSION MATRIX & t-SNE
# =====================================================================

print_ascii(
    "EVALUATION MODULE",
    r"""
LOSO predictions
      │
      ├── per-subject accuracy
      ├── mean ± std
      ├── confusion matrix
      │
      └── CRNN-DF feature vectors v
               │
               ▼
             t-SNE
               │
               ▼
  before/after style feature visualization
  similar subjects/classes should converge/separate
"""
)

def summarize_results(results_df):
    if results_df is None:
        return None
    acc = np.asarray(results_df["accuracy"], dtype=float)
    print(results_df)
    print("\nMean accuracy: {:.2f}%".format(acc.mean() * 100))
    print("Std accuracy : {:.2f}%".format(acc.std(ddof=0) * 100))
    return acc.mean(), acc.std(ddof=0)

def plot_loso_accuracy(results_df):
    if results_df is None:
        return
    plt.figure(figsize=(8, 4))
    plt.bar(results_df["subject"], results_df["accuracy"] * 100)
    plt.axhline(25.0, linestyle="--", linewidth=1.5, label="4-class chance")
    plt.ylabel("Accuracy (%)")
    plt.xlabel("LOSO target subject")
    plt.title("LOSO subject-independent accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_confusion(all_predictions):
    if not all_predictions:
        print("No predictions available.")
        return
    y_true = np.concatenate([d["y_true"] for d in all_predictions])
    y_pred = np.concatenate([d["y_pred"] for d in all_predictions])
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(N_CLASSES))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("LOSO normalized confusion matrix")
    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

def plot_tsne(all_features, perplexity=20, random_state=SEED):
    if not all_features:
        print("No features available.")
        return

    Xf = np.concatenate([d["features"] for d in all_features], axis=0)
    yf = np.concatenate([d["labels"] for d in all_features], axis=0)
    sf = np.concatenate([
        np.full(len(d["labels"]), d["subject"], dtype=np.int64)
        for d in all_features
    ])

    # t-SNE requires perplexity < n_samples.
    p = min(perplexity, max(2, (len(Xf) - 1) // 3))
    p = min(p, len(Xf) - 1)

    tsne = TSNE(
        n_components=2,
        perplexity=p,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
    )
    X2 = tsne.fit_transform(Xf)

    plt.figure(figsize=(9, 7))
    for cls in range(N_CLASSES):
        idx = yf == cls
        plt.scatter(
            X2[idx, 0],
            X2[idx, 1],
            s=14,
            alpha=0.65,
            label=CLASS_NAMES[cls],
        )
    plt.title("CRNN-DF discriminative feature t-SNE")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Subject-separated visualization, analogous to the paper's qualitative
    # feature distribution figures.
    subjects = np.unique(sf)
    cols = min(3, len(subjects))
    rows = int(np.ceil(len(subjects) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, subject in zip(axes, subjects):
        for cls in range(N_CLASSES):
            idx = (sf == subject) & (yf == cls)
            ax.scatter(X2[idx, 0], X2[idx, 1], s=10, alpha=0.55)
        ax.set_title(f"Subject A{subject}")
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
    for ax in axes[len(subjects):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# ---------------------- Optional result inspection --------------------

if RUN_SMOKE_EXPERIMENT:
    summarize_results(smoke_results)
    plot_loso_accuracy(smoke_results)
    plot_confusion(smoke_predictions)
    plot_tsne(smoke_features, perplexity=8)

# --------------------------- Reference table --------------------------
#
# Source paper Table 3:
# CRNN-DF, no generated augmentation:
# A1 65.51, A2 45.18, A3 78.62, A4 53.58, A5 55.64,
# A6 56.03, A7 71.28, A8 75.02, A9 70.78
# Mean 63.52, Std 10.70
#
# Source paper Table 4:
# Naug=500  -> 68.53±10.55
# Naug=1000 -> 69.90±10.97
# Naug=2000 -> 71.31±11.42
# Naug=3000 -> 72.82±10.44
# Naug=4000 -> 72.82±10.94
#
# The abstract reports 72.74±10.44. The notebook intentionally displays
# the table values without inventing a reconciliation.

print("Notebook complete.")
print("For the exact paper-scale run, load all 9 subjects and set:")
print("  RUN_FULL_EXPERIMENT=True")
print("  n_aug=3000  # 750 per class")
print("  gan_epochs=50 (or your available compute budget)")
print("  classifier_epochs=100")
